<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/day3_aqueduct_hazard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# — Setup
!pip install earthengine-api geemap geopandas folium -q

import ee
import geemap
import pandas as pd
import geopandas as gpd
import folium

ee.Authenticate()
ee.Initialize(project='tamil-nadu-flood-risk')

print(ee.String('Day 3 — fresh start ready!').getInfo())

Day 3 — fresh start ready!


In [14]:
# School data
import pandas as pd
import ee

ee.Initialize(project='tamil-nadu-flood-risk')

# School data
data = {
    'school_name': [
        'Govt High School Chennai', 'Govt School Cuddalore',
        'Panchayat School Nagapattinam', 'Govt School Thanjavur',
        'High School Coimbatore', 'Govt School Madurai',
        'Panchayat School Tirunelveli', 'Govt School Salem',
        'High School Vellore', 'Govt School Tiruchirappalli',
        'School Ramanathapuram', 'School Puducherry Border',
        'School Kanchipuram', 'School Villupuram',
        'School Tuticorin'
    ],
    'latitude': [
        13.08, 11.75, 10.76, 10.78, 11.01,
        9.93, 8.71, 11.65, 12.92, 10.79,
        9.37, 11.93, 12.83, 11.93, 8.80
    ],
    'longitude': [
        80.27, 79.75, 79.84, 79.13, 76.96,
        78.12, 77.69, 78.16, 79.13, 78.68,
        78.83, 79.83, 79.70, 79.49, 78.15
    ],
    'connectivity': [
        'connected', 'none', 'none',
        'connected', 'connected', 'connected',
        'none', 'connected', 'connected',
        'none', 'none', 'connected',
        'none', 'none', 'connected'
    ]
}

schools_df = pd.DataFrame(data)
print(f'✅ {len(schools_df)} schools loaded')
print(schools_df[['school_name', 'connectivity']].to_string(index=False))

✅ 15 schools loaded
                  school_name connectivity
     Govt High School Chennai    connected
        Govt School Cuddalore         none
Panchayat School Nagapattinam         none
        Govt School Thanjavur    connected
       High School Coimbatore    connected
          Govt School Madurai    connected
 Panchayat School Tirunelveli         none
            Govt School Salem    connected
          High School Vellore    connected
  Govt School Tiruchirappalli         none
        School Ramanathapuram         none
     School Puducherry Border    connected
           School Kanchipuram         none
            School Villupuram         none
             School Tuticorin    connected


In [15]:
# School data
import pandas as pd
import ee

ee.Initialize(project='tamil-nadu-flood-risk')

# School data
data = {
    'school_name': [
        'Govt High School Chennai', 'Govt School Cuddalore',
        'Panchayat School Nagapattinam', 'Govt School Thanjavur',
        'High School Coimbatore', 'Govt School Madurai',
        'Panchayat School Tirunelveli', 'Govt School Salem',
        'High School Vellore', 'Govt School Tiruchirappalli',
        'School Ramanathapuram', 'School Puducherry Border',
        'School Kanchipuram', 'School Villupuram',
        'School Tuticorin'
    ],
    'latitude': [
        13.08, 11.75, 10.76, 10.78, 11.01,
        9.93, 8.71, 11.65, 12.92, 10.79,
        9.37, 11.93, 12.83, 11.93, 8.80
    ],
    'longitude': [
        80.27, 79.75, 79.84, 79.13, 76.96,
        78.12, 77.69, 78.16, 79.13, 78.68,
        78.83, 79.83, 79.70, 79.49, 78.15
    ],
    'connectivity': [
        'connected', 'none', 'none',
        'connected', 'connected', 'connected',
        'none', 'connected', 'connected',
        'none', 'none', 'connected',
        'none', 'none', 'connected'
    ]
}

schools_df = pd.DataFrame(data)
print(f'✅ {len(schools_df)} schools loaded')
print(schools_df[['school_name', 'connectivity']].to_string(index=False))

✅ 15 schools loaded
                  school_name connectivity
     Govt High School Chennai    connected
        Govt School Cuddalore         none
Panchayat School Nagapattinam         none
        Govt School Thanjavur    connected
       High School Coimbatore    connected
          Govt School Madurai    connected
 Panchayat School Tirunelveli         none
            Govt School Salem    connected
          High School Vellore    connected
  Govt School Tiruchirappalli         none
        School Ramanathapuram         none
     School Puducherry Border    connected
           School Kanchipuram         none
            School Villupuram         none
             School Tuticorin    connected


In [18]:
# Debug flood database
import ee
ee.Initialize(project='tamil-nadu-flood-risk')

tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

# Check GFD collection
gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')

try:
    size = gfd.size().getInfo()
    print('✅ GFD loaded — total events:', size)
except Exception as e:
    print('❌ GFD error:', str(e))

# Check India filter
try:
    india_floods = gfd.filterBounds(tamil_nadu)
    india_size = india_floods.size().getInfo()
    print('✅ Tamil Nadu events:', india_size)
except Exception as e:
    print('❌ Filter error:', str(e))

✅ GFD loaded — total events: 913
✅ Tamil Nadu events: 37


In [17]:
# Display flood frequency map
import ee
import geemap

ee.Initialize(project='tamil-nadu-flood-risk')

tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

# Build flood frequency map
gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
flood_frequency = gfd.select('flooded').sum().clip(tamil_nadu)

# Build school FeatureCollection
import pandas as pd
data = {
    'school_name': [
        'Govt High School Chennai', 'Govt School Cuddalore',
        'Panchayat School Nagapattinam', 'Govt School Thanjavur',
        'High School Coimbatore', 'Govt School Madurai',
        'Panchayat School Tirunelveli', 'Govt School Salem',
        'High School Vellore', 'Govt School Tiruchirappalli',
        'School Ramanathapuram', 'School Puducherry Border',
        'School Kanchipuram', 'School Villupuram',
        'School Tuticorin'
    ],
    'latitude': [
        13.08, 11.75, 10.76, 10.78, 11.01,
        9.93, 8.71, 11.65, 12.92, 10.79,
        9.37, 11.93, 12.83, 11.93, 8.80
    ],
    'longitude': [
        80.27, 79.75, 79.84, 79.13, 76.96,
        78.12, 77.69, 78.16, 79.13, 78.68,
        78.83, 79.83, 79.70, 79.49, 78.15
    ],
    'connectivity': [
        'connected', 'none', 'none',
        'connected', 'connected', 'connected',
        'none', 'connected', 'connected',
        'none', 'none', 'connected',
        'none', 'none', 'connected'
    ]
}
schools_df = pd.DataFrame(data)

features = []
for _, row in schools_df.iterrows():
    features.append(ee.Feature(
        ee.Geometry.Point([row['longitude'], row['latitude']]),
        {'name': row['school_name'], 'connectivity': row['connectivity']}
    ))
schools_ee = ee.FeatureCollection(features)

# Map
Map = geemap.Map()
Map.centerObject(tamil_nadu, zoom=7)

# Flood frequency layer
Map.addLayer(
    flood_frequency,
    {
        'min': 0, 'max': 10,
        'palette': ['white', 'lightblue', '2196F3', '0d47a1', '021C3B']
    },
    'Flood frequency 2000-2018'
)

# Connected schools — green
connected = schools_ee.filter(ee.Filter.eq('connectivity', 'connected'))
Map.addLayer(connected, {'color': '00cc00'}, 'Connected schools')

# No connectivity — red
no_conn = schools_ee.filter(ee.Filter.eq('connectivity', 'none'))
Map.addLayer(no_conn, {'color': 'ff0000'}, 'No connectivity schools')

# Display
display(Map)

Map(center=[10.749705958455543, 78.24999999999999], controls=(WidgetControl(options=['position', 'transparent_…

In [19]:
# Extract flood frequency at each school
import ee
import pandas as pd

ee.Initialize(project='tamil-nadu-flood-risk')

tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

# Rebuild flood frequency
gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
flood_frequency = gfd.select('flooded').sum().clip(tamil_nadu)

# Build buffered school features
data = {
    'school_name': [
        'Govt High School Chennai', 'Govt School Cuddalore',
        'Panchayat School Nagapattinam', 'Govt School Thanjavur',
        'High School Coimbatore', 'Govt School Madurai',
        'Panchayat School Tirunelveli', 'Govt School Salem',
        'High School Vellore', 'Govt School Tiruchirappalli',
        'School Ramanathapuram', 'School Puducherry Border',
        'School Kanchipuram', 'School Villupuram',
        'School Tuticorin'
    ],
    'latitude': [
        13.08, 11.75, 10.76, 10.78, 11.01,
        9.93, 8.71, 11.65, 12.92, 10.79,
        9.37, 11.93, 12.83, 11.93, 8.80
    ],
    'longitude': [
        80.27, 79.75, 79.84, 79.13, 76.96,
        78.12, 77.69, 78.16, 79.13, 78.68,
        78.83, 79.83, 79.70, 79.49, 78.15
    ],
    'connectivity': [
        'connected', 'none', 'none',
        'connected', 'connected', 'connected',
        'none', 'connected', 'connected',
        'none', 'none', 'connected',
        'none', 'none', 'connected'
    ]
}
schools_df = pd.DataFrame(data)

# Build 1km buffered features
features = []
for _, row in schools_df.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    buffer = point.buffer(1000)
    features.append(ee.Feature(buffer, {
        'school_name': row['school_name'],
        'connectivity': row['connectivity']
    }))
schools_buffered = ee.FeatureCollection(features)

# Extract mean flood frequency per school buffer
flood_stats = flood_frequency.reduceRegions(
    collection=schools_buffered,
    reducer=ee.Reducer.mean(),
    scale=250
)

# Parse results
data_out = [f['properties'] for f in flood_stats.getInfo()['features']]
gfd_df = pd.DataFrame(data_out)
gfd_df = gfd_df.rename(columns={'mean': 'flood_frequency'})
gfd_df['flood_frequency'] = gfd_df['flood_frequency'].fillna(0).round(2)

# Risk tier
def gfd_risk(freq):
    if freq >= 3:
        return 'HIGH'
    elif freq >= 1:
        return 'MEDIUM'
    else:
        return 'LOW'

gfd_df['gfd_risk'] = gfd_df['flood_frequency'].apply(gfd_risk)
gfd_df = gfd_df.sort_valu

AttributeError: 'DataFrame' object has no attribute 'sort_valu'

In [20]:
# Fixed sorting and display
gfd_df = gfd_df.sort_values('flood_frequency', ascending=False).reset_index(drop=True)

print('=== FLOOD FREQUENCY 2000-2018 — TAMIL NADU SCHOOLS ===')
print('(Number of flood events detected by MODIS satellite)\n')
for _, row in gfd_df.iterrows():
    conn = '⚠' if row['connectivity'] == 'none' else ' '
    print(f"{conn} {row['school_name'][:35]:<35} | "
          f"Floods: {row['flood_frequency']:4.1f} | "
          f"Risk: {row['gfd_risk']}")

=== FLOOD FREQUENCY 2000-2018 — TAMIL NADU SCHOOLS ===
(Number of flood events detected by MODIS satellite)

  School Puducherry Border            | Floods:  1.0 | Risk: LOW
⚠ Panchayat School Nagapattinam       | Floods:  0.6 | Risk: LOW
  School Tuticorin                    | Floods:  0.1 | Risk: LOW
⚠ Panchayat School Tirunelveli        | Floods:  0.0 | Risk: LOW
  Govt School Thanjavur               | Floods:  0.0 | Risk: LOW
⚠ Govt School Cuddalore               | Floods:  0.0 | Risk: LOW
  Govt High School Chennai            | Floods:  0.0 | Risk: LOW
  Govt School Madurai                 | Floods:  0.0 | Risk: LOW
  High School Coimbatore              | Floods:  0.0 | Risk: LOW
  Govt School Salem                   | Floods:  0.0 | Risk: LOW
  High School Vellore                 | Floods:  0.0 | Risk: LOW
⚠ School Ramanathapuram               | Floods:  0.0 | Risk: LOW
⚠ Govt School Tiruchirappalli         | Floods:  0.0 | Risk: LOW
⚠ School Kanchipuram                  | Floods

In [22]:
# Cell 6 — Fixed master risk table
import pandas as pd

# Day 1 SAR data
sar_data = {
    'school_name': [
        'School Puducherry Border',
        'School Ramanathapuram',
        'Govt School Cuddalore',
        'Panchayat School Nagapattinam',
        'Govt School Madurai',
        'High School Vellore',
        'School Villupuram',
        'School Tuticorin',
        'Govt School Tiruchirappalli',
        'Govt School Thanjavur',
        'Govt High School Chennai',
        'Panchayat School Tirunelveli',
        'High School Coimbatore',
        'Govt School Salem',
        'School Kanchipuram'
    ],
    'connectivity': [
        'connected', 'none', 'none', 'none',
        'connected', 'connected', 'none', 'connected',
        'none', 'connected', 'connected', 'none',
        'connected', 'connected', 'none'
    ],
    'sar_flood_pct': [
        10.87, 1.59, 1.56, 1.07, 0.95,
        0.63, 0.63, 0.34, 0.34, 0.00,
        0.00, 0.00, 0.00, 0.00, 0.00
    ]
}

# Day 2 JRC data
jrc_data = {
    'school_name': [
        'School Puducherry Border',
        'School Tuticorin',
        'Panchayat School Nagapattinam',
        'Govt School Thanjavur',
        'School Ramanathapuram',
        'High School Vellore',
        'Govt School Cuddalore',
        'School Kanchipuram',
        'Govt School Tiruchirappalli',
        'Panchayat School Tirunelveli',
        'Govt High School Chennai',
        'Govt School Madurai',
        'School Villupuram',
        'High School Coimbatore',
        'Govt School Salem'
    ],
    'water_occurrence_pct': [
        97.20, 60.95, 44.96, 43.46, 38.81,
        32.90, 30.18, 29.21, 25.28, 19.47,
        11.91, 9.79, 2.76, 0.00, 0.00
    ]
}

# Day 3 GFD data
gfd_data = gfd_df[['school_name', 'flood_frequency']].copy()

# Merge all three
sar_df = pd.DataFrame(sar_data)
jrc_df = pd.DataFrame(jrc_data)

master = sar_df \
    .merge(jrc_df, on='school_name') \
    .merge(gfd_data, on='school_name')

# Normalise each component 0-100
master['sar_norm'] = (
    master['sar_flood_pct'] /
    master['sar_flood_pct'].max() * 100
).round(1)

master['jrc_norm'] = master['water_occurrence_pct'].round(1)

master['gfd_norm'] = (
    master['flood_frequency'] /
    max(master['flood_frequency'].max(), 1) * 100
).round(1)

# Weighted master score
# JRC 50% + SAR 30% + GFD 20%
master['master_score'] = (
    0.50 * master['jrc_norm'] +
    0.30 * master['sar_norm'] +
    0.20 * master['gfd_norm']
).round(1)

# Connectivity penalty
master['conn_penalty'] = master['connectivity'].apply(
    lambda x: 15 if x == 'none' else 0
)
master['final_score'] = (
    master['master_score'] + master['conn_penalty']
).round(1)

# Risk tier
def final_risk(score):
    if score >= 50:
        return 'CRITICAL'
    elif score >= 25:
        return 'HIGH'
    elif score >= 10:
        return 'MEDIUM'
    else:
        return 'LOW'

master['final_risk'] = master['final_score'].apply(final_risk)
master = master.sort_values(
    'final_score', ascending=False
).reset_index(drop=True)
master['rank'] = master.index + 1

# Print results
print('=== MASTER RISK TABLE — 3 DATA SOURCES ===')
print('JRC(50%) + SAR(30%) + GFD(20%) + Connectivity\n')
print(f"{'#':<3} {'School':<32} {'Conn':<5} "
      f"{'SAR':>5} {'JRC':>6} {'GFD':>5} "
      f"{'Score':>6} {'Risk'}")
print('-' * 75)
for _, row in master.iterrows():
    conn = 'No' if row['connectivity'] == 'none' else 'Yes'
    print(
        f"#{int(row['rank']):<2} "
        f"{row['school_name'][:31]:<32} "
        f"{conn:<5} "
        f"{row['sar_norm']:>5.1f} "
        f"{row['jrc_norm']:>6.1f} "
        f"{row['gfd_norm']:>5.1f} "
        f"{row['final_score']:>6.1f} "
        f"{row['final_risk']}"
    )

=== MASTER RISK TABLE — 3 DATA SOURCES ===
JRC(50%) + SAR(30%) + GFD(20%) + Connectivity

#   School                           Conn    SAR    JRC   GFD  Score Risk
---------------------------------------------------------------------------
#1  School Puducherry Border         Yes   100.0   97.2  98.0   98.2 CRITICAL
#2  Panchayat School Nagapattinam    No      9.8   45.0  58.0   52.0 CRITICAL
#3  School Ramanathapuram            No     14.6   38.8   0.0   38.8 HIGH
#4  Govt School Cuddalore            No     14.4   30.2   0.0   34.4 HIGH
#5  School Tuticorin                 Yes     3.1   61.0  13.0   34.0 HIGH
#6  School Kanchipuram               No      0.0   29.2   0.0   29.6 HIGH
#7  Govt School Tiruchirappalli      No      3.1   25.3   0.0   28.6 HIGH
#8  Panchayat School Tirunelveli     No      0.0   19.5   2.0   25.2 HIGH
#9  Govt School Thanjavur            Yes     0.0   43.5   0.0   21.8 MEDIUM
#10 High School Vellore              Yes     5.8   32.9   0.0   18.2 MEDIUM
#11 Scho

In [23]:
# Final master risk map
import folium
import pandas as pd

# Add coordinates
coords = {
    'School Puducherry Border':      [11.93, 79.83],
    'Panchayat School Nagapattinam': [10.76, 79.84],
    'School Ramanathapuram':         [9.37,  78.83],
    'Govt School Cuddalore':         [11.75, 79.75],
    'School Tuticorin':              [8.80,  78.15],
    'School Kanchipuram':            [12.83, 79.70],
    'Govt School Tiruchirappalli':   [10.79, 78.68],
    'Panchayat School Tirunelveli':  [8.71,  77.69],
    'Govt School Thanjavur':         [10.78, 79.13],
    'High School Vellore':           [12.92, 79.13],
    'School Villupuram':             [11.93, 79.49],
    'Govt School Madurai':           [9.93,  78.12],
    'Govt High School Chennai':      [13.08, 80.27],
    'High School Coimbatore':        [11.01, 76.96],
    'Govt School Salem':             [11.65, 78.16],
}

master['latitude']  = master['school_name'].map(lambda x: coords[x][0])
master['longitude'] = master['school_name'].map(lambda x: coords[x][1])

# Build map
m = folium.Map(
    location=[10.5, 78.5],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Colour by risk
def get_color(risk):
    return {
        'CRITICAL': '#cc0000',
        'HIGH':     '#ff6600',
        'MEDIUM':   '#ffaa00',
        'LOW':      '#2d8a4e'
    }.get(risk, 'gray')

# Radius by score
def get_radius(score):
    if score >= 50: return 20
    elif score >= 25: return 15
    elif score >= 10: return 10
    else: return 7

# Add schools
for _, row in master.iterrows():
    color  = get_color(row['final_risk'])
    radius = get_radius(row['final_score'])
    conn   = '⚠ No connectivity' if row['connectivity'] == 'none' else '✅ Connected'

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        tooltip=f"#{int(row['rank'])} {row['school_name']}",
        popup=folium.Popup(
            f"""
            <b>#{int(row['rank'])} {row['school_name']}</b><br>
            <hr style='margin:4px 0'>
            🛰 <b>SAR 2023:</b> {row['sar_norm']}%<br>
            📅 <b>JRC History:</b> {row['jrc_norm']}%<br>
            🌊 <b>GFD Events:</b> {row['gfd_norm']}%<br>
            📊 <b>Master score:</b> {row['final_score']}<br>
            📡 <b>Connectivity:</b> {conn}<br>
            <hr style='margin:4px 0'>
            🎯 <b>Risk: {row['final_risk']}</b>
            """,
            max_width=280
        )
    ).add_to(m)

    # Rank label
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;'
                 f'color:white;text-align:center;margin-top:4px">'
                 f'#{int(row["rank"])}</div>',
            icon_size=(20, 20),
            icon_anchor=(10, 10)
        )
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;
     background:white;padding:14px;border-radius:10px;
     border:1px solid #ccc;font-size:12px;z-index:1000;
     box-shadow:2px 2px 6px rgba(0,0,0,0.15)">
     <b>🏫 Master Flood Risk Score</b><br>
     <b>SAR + JRC + GFD (3 sources)</b><br><br>
     <span style='color:#cc0000;font-size:16px'>●</span> Critical (≥ 50)<br>
     <span style='color:#ff6600;font-size:16px'>●</span> High (25–50)<br>
     <span style='color:#ffaa00;font-size:16px'>●</span> Medium (10–25)<br>
     <span style='color:#2d8a4e;font-size:16px'>●</span> Low (< 10)<br><br>
     <i>Size = risk score magnitude</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save
m.save('tamil_nadu_master_risk_map.html')
master.to_csv('tamil_nadu_master_risk_scores.csv', index=False)
print('✅ Master risk map saved!')
print('✅ Master risk CSV saved!')
display(m)

✅ Master risk map saved!
✅ Master risk CSV saved!


In [26]:
# Cell 8 — Download all files
from google.colab import files

files.download('tamil_nadu_master_risk_map.html')
files.download('tamil_nadu_master_risk_scores.csv')
print('✅ All files downloaded!')



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!
